# Gerar Ground Truth de PII via Gemini — conjunto de teste holdout

Espelha `notebooks/01.gerar_ground_truth_gemini.ipynb`, mas aponta para `data/documento_social/teste_holdout/arquivos/` — os documentos genuinamente novos, fora do treino e da validação usados nos notebooks 01-06. Envia cada um ao Gemini usando o mesmo prompt e schema estruturado em `prompts/`, e salva o gabarito (JSON por documento) em `data/documento_social/teste_holdout/gabarito_gemini/<modelo>/`.

Sem este gabarito o holdout só permite inspeção visual das previsões (ver notebook 11) — com ele, o notebook 09 (alinhamento) e o notebook 10 (métricas) conseguem medir precision/recall/F1 de verdade sobre documentos que o modelo nunca viu, em vez de só o split interno treino/teste que o notebook 06 já mede.

**Pré-requisitos:**
- `pip install google-genai python-dotenv`
- Variável de ambiente `GEMINI_API_KEY` definida (ou um arquivo `.env` na raiz do projeto com `GEMINI_API_KEY=...`)
- Documentos já colocados em `data/documento_social/teste_holdout/arquivos/` (ver `scripts/selecionar_amostra_teste_holdout.py`)

In [1]:
import datetime
import json
import re
import shutil
import tempfile
import time
import unicodedata
from pathlib import Path

from google import genai
from google.genai import types

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## Configuração de caminhos

Resolve a raiz do projeto automaticamente (procura `requirements.txt` subindo a partir do diretório atual), para o notebook funcionar independentemente de ser executado a partir de `notebooks/` ou da raiz.

In [2]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

PASTA_ENTRADA = RAIZ / "data" / "documento_social" / "teste_holdout" / "arquivos"
PASTA_SAIDA_BASE = RAIZ / "data" / "documento_social" / "teste_holdout" / "gabarito_gemini"
PASTA_PROMPTS = RAIZ / "prompts"

print(f"Raiz do projeto: {RAIZ}")
print(f"Entrada (holdout): {PASTA_ENTRADA}")
print(f"Saída (base):      {PASTA_SAIDA_BASE}")
print(f"Prompts:           {PASTA_PROMPTS}")

Raiz do projeto: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3
Entrada (holdout): d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\arquivos
Saída (base):      d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\gabarito_gemini
Prompts:           d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\prompts


## Carrega o prompt de sistema e o schema estruturado

Mesmos arquivos de `prompts/` usados no notebook 01, para manter o holdout anotado com exatamente o mesmo critério do gabarito de treino.

In [3]:
SYSTEM_INSTRUCTION = (PASTA_PROMPTS / "pii_ground_truth_system_prompt.txt").read_text(encoding="utf-8")
RESPONSE_SCHEMA = json.loads((PASTA_PROMPTS / "pii_ground_truth_response_schema.json").read_text(encoding="utf-8"))

print(SYSTEM_INSTRUCTION[:300], "...")

Você é um anotador de dados sensíveis (PII) em documentos brasileiros, para fins de \
pesquisa acadêmica sobre proteção de dados (LGPD).

Sua tarefa é identificar TODAS as ocorrências de:
- CPF (Cadastro de Pessoas Físicas). Inclua aqui também ocorrências de CIN (Carteira de
  Identidade Nacional, o ...


## Parâmetros do experimento

`MODEL` deve idealmente bater com o modelo usado para o gabarito de treino (notebook 01), para não introduzir uma diferença de critério de anotação entre treino e holdout.

In [4]:
MODEL = "gemini-2.5-flash"  # deve bater com o MODELO_GABARITO usado no notebook 09 (alinhamento)
SOBRESCREVER = False        # True para reprocessar arquivos que já têm gabarito salvo

EXTENSOES_SUPORTADAS = {".pdf", ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

PASTA_SAIDA = PASTA_SAIDA_BASE / MODEL
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Modelo: {MODEL}")
print(f"Saída:  {PASTA_SAIDA}")

Modelo: gemini-2.5-flash
Saída:  d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\gabarito_gemini\gemini-2.5-flash


## Tabela de preços (USD por 1M de tokens)

**IMPORTANTE:** preços da API do Gemini mudam com o tempo — confira em https://ai.google.dev/pricing antes de rodar um lote grande, se o custo for uma preocupação real.

In [5]:
# preço por 1.000.000 de tokens, em USD
PRECOS = {
    "gemini-2.5-pro": {"entrada": 1.25, "saida": 10.00, "entrada_acima_200k": 2.50, "saida_acima_200k": 15.00},
    "gemini-2.5-flash": {"entrada": 0.30, "saida": 2.50},
    "gemini-2.5-flash-lite": {"entrada": 0.10, "saida": 0.40},
}


def calcular_custo(model: str, tokens_entrada: int, tokens_saida: int) -> float:
    precos = PRECOS.get(model)
    if precos is None:
        return float("nan")
    preco_entrada = precos["entrada"]
    preco_saida = precos["saida"]
    if tokens_entrada > 200_000 and "entrada_acima_200k" in precos:
        preco_entrada = precos["entrada_acima_200k"]
        preco_saida = precos["saida_acima_200k"]
    custo = (tokens_entrada / 1_000_000) * preco_entrada + (tokens_saida / 1_000_000) * preco_saida
    return custo

In [6]:
import os
import sys

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    sys.exit("Defina GEMINI_API_KEY (variável de ambiente ou arquivo .env na raiz do projeto) antes de continuar.")

client = genai.Client(api_key=api_key)

## Função de processamento de um arquivo

**Nota sobre nomes de arquivo com acento:** o SDK do Gemini (File API) falha ao enviar arquivos cujo nome contém caracteres não-ASCII — fazemos uma cópia temporária com nome seguro em ASCII antes de enviar, e apagamos a cópia depois (mesma lógica do notebook 01).

In [7]:
def nome_ascii_seguro(nome: str) -> str:
    limpo = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    limpo = re.sub(r"[^A-Za-z0-9._-]+", "_", limpo).strip("_")
    return limpo or "documento"


def processar_arquivo(caminho: Path, model: str) -> tuple[dict, dict]:
    """Envia um arquivo ao Gemini via File API. Retorna (entidades, info_uso_tokens)."""
    nome_seguro = nome_ascii_seguro(caminho.stem) + caminho.suffix.lower()

    with tempfile.TemporaryDirectory() as tmp_dir:
        caminho_tmp = Path(tmp_dir) / nome_seguro
        shutil.copyfile(caminho, caminho_tmp)

        arquivo_remoto = client.files.upload(file=str(caminho_tmp))

        while arquivo_remoto.state.name == "PROCESSING":
            time.sleep(2)
            arquivo_remoto = client.files.get(name=arquivo_remoto.name)

    if arquivo_remoto.state.name == "FAILED":
        raise RuntimeError(f"Falha ao processar arquivo no Gemini: {caminho.name}")

    resposta = client.models.generate_content(
        model=model,
        contents=[
            arquivo_remoto,
            "Analise este documento e retorne todas as ocorrências de CPF, RG, CIN, NOME e ENDERECO "
            "conforme as instruções do sistema.",
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            temperature=0,
        ),
    )

    dados = json.loads(resposta.text)

    uso = resposta.usage_metadata
    tokens_entrada = getattr(uso, "prompt_token_count", 0) or 0
    tokens_saida = getattr(uso, "candidates_token_count", 0) or 0
    tokens_total = getattr(uso, "total_token_count", tokens_entrada + tokens_saida) or (tokens_entrada + tokens_saida)
    custo_usd = calcular_custo(model, tokens_entrada, tokens_saida)

    info_uso = {
        "tokens_entrada": tokens_entrada,
        "tokens_saida": tokens_saida,
        "tokens_total": tokens_total,
        "custo_usd": round(custo_usd, 5),
    }

    try:
        client.files.delete(name=arquivo_remoto.name)
    except Exception:
        pass

    return dados, info_uso

## Loop principal — processa todos os arquivos do holdout

In [8]:
import concurrent.futures

PARALELISMO = 10  # quantas requisições ao Gemini rodam em paralelo (baixe se começar a receber erro 429)

arquivos = sorted(
    p for p in PASTA_ENTRADA.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSOES_SUPORTADAS
)

if not arquivos:
    raise SystemExit(f"Nenhum arquivo suportado encontrado em {PASTA_ENTRADA}")

print(f"Encontrados {len(arquivos)} arquivo(s) em {PASTA_ENTRADA}")
print(f"Modelo: {MODEL} | Paralelismo: {PARALELISMO}")

arquivos_pendentes = []
for caminho in arquivos:
    destino = PASTA_SAIDA / f"{caminho.stem}.json"
    if destino.exists() and not SOBRESCREVER:
        print(f"{caminho.name}: já existe, pulando (SOBRESCREVER=True para refazer)")
        continue
    arquivos_pendentes.append(caminho)

print(f"\n{len(arquivos_pendentes)} arquivo(s) a processar.\n")


def processar_e_salvar(caminho: Path) -> dict:
    """Processa um arquivo e salva o resultado. Roda em uma thread do pool (I/O-bound: upload + espera do Gemini)."""
    destino = PASTA_SAIDA / f"{caminho.stem}.json"
    try:
        dados, info_uso = processar_arquivo(caminho, MODEL)
    except Exception as e:
        return {"caminho": caminho, "erro": str(e)}

    registro = {
        "document_id": caminho.stem,
        "source_file": caminho.name,
        "annotated_by": MODEL,
        "annotated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "usage": info_uso,
        "entities": dados.get("entities", []),
    }
    destino.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"caminho": caminho, "n_entidades": len(registro["entities"]), "info_uso": info_uso}


custo_total_usd = 0.0
concluidos = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=PARALELISMO) as executor:
    futuros = {executor.submit(processar_e_salvar, caminho): caminho for caminho in arquivos_pendentes}
    for futuro in concurrent.futures.as_completed(futuros):
        caminho = futuros[futuro]
        resultado = futuro.result()
        concluidos += 1

        if "erro" in resultado:
            print(f"[{concluidos}/{len(arquivos_pendentes)}] {caminho.name}: ERRO -> {resultado['erro']}")
            continue

        info_uso = resultado["info_uso"]
        custo_total_usd += info_uso["custo_usd"]
        print(
            f"[{concluidos}/{len(arquivos_pendentes)}] {caminho.name}: {resultado['n_entidades']} entidade(s) | "
            f"tokens entrada={info_uso['tokens_entrada']} saida={info_uso['tokens_saida']} | "
            f"custo=US$ {info_uso['custo_usd']:.5f}"
        )

print(f"\nConcluído. Custo total estimado desta execução: US$ {custo_total_usd:.4f}")
print("Próximo passo: notebooks/08.extrair_texto_holdout.ipynb (kernel locscan-ocr), depois notebooks/09.alinhar_ground_truth_holdout.ipynb.")

Encontrados 30 arquivo(s) em d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\arquivos
Modelo: gemini-2.5-flash | Paralelismo: 10
cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm1ow1j3e0008uiqd53mf55j4-CONTRATO SOCIAL NOVO 22.03.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm1qczv5u0008v6qd426f9fgv-CORTEC - Alteração Consolidada290720.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm2j3nwn9001qgpqd51wm07ce-Alteração Consolidada setembro14.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm2t2y1fg00b12yqde39j7bwb-Alteração Consolidada 102019.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm3376wbs002xzaqd1p4r02h0-15 Alteração Contratual.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cm37fqf0n0006hnqd6jjm3x

## Conferência rápida do resultado (entidades e custo acumulado)

In [9]:
gabaritos = sorted(PASTA_SAIDA.glob("*.json"))
print(f"{len(gabaritos)} gabarito(s) salvo(s) em {PASTA_SAIDA}\n")

custo_acumulado = 0.0
for g in gabaritos:
    dados = json.loads(g.read_text(encoding="utf-8"))
    contagem = {}
    for e in dados["entities"]:
        contagem[e["type"]] = contagem.get(e["type"], 0) + 1
    custo = dados.get("usage", {}).get("custo_usd")
    if custo is not None:
        custo_acumulado += custo
    custo_str = f"US$ {custo:.5f}" if custo is not None else "N/A (gabarito antigo, sem usage)"
    print(f"{g.name}: {contagem} | custo: {custo_str}")

print(f"\nCusto acumulado (gabaritos com info de uso disponível): US$ {custo_acumulado:.4f}")

30 gabarito(s) salvo(s) em d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\gabarito_gemini\gemini-2.5-flash

cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf.json: {'NOME': 38, 'CPF': 9, 'RG': 2, 'ENDERECO': 2} | custo: US$ 0.01413
cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.json: {'NOME': 15, 'CPF': 5, 'RG': 2, 'ENDERECO': 2} | custo: US$ 0.00657
cm1ow1j3e0008uiqd53mf55j4-CONTRATO SOCIAL NOVO 22.03.pdf.json: {'NOME': 9, 'RG': 2, 'CPF': 2, 'ENDERECO': 2} | custo: US$ 0.00504
cm1qczv5u0008v6qd426f9fgv-CORTEC - Alteração Consolidada290720.pdf.json: {'NOME': 41, 'CPF': 16, 'RG': 3, 'ENDERECO': 3} | custo: US$ 0.01990
cm2j3nwn9001qgpqd51wm07ce-Alteração Consolidada setembro14.pdf.json: {'NOME': 24, 'RG': 3, 'CPF': 3, 'ENDERECO': 3} | custo: US$ 0.00951
cm2t2y1fg00b12yqde39j7bwb-Alteração Consolidada 102019.pdf.json: {'NOME': 34, 'CPF': 8, 'ENDERECO': 1, 'RG': 1} | custo: US$ 0.01330
cm3376wbs002xzaqd1p4r02h0-15 Alteração Contratual.pd